In [1]:
import pandas as pd
import json
import weaviate
from sentence_transformers import SentenceTransformer


d:\Users\Daniel Hamill\Documents\Projects\discord-llm-agent\history-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MESSAGES_SRC = "./exported_messages.csv"

# Model is only needed to encode the query at search time.
model = SentenceTransformer("BAAI/bge-base-en-v1.5")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4267.89it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
df = pd.read_csv(MESSAGES_SRC)
df["embedding"] = df["embedding"].apply(json.loads)

print(f"Loaded {len(df)} messages")
df.head()


Loaded 8228 messages


,id,user,content,timestamp,embedding
0,1490440429215158272,tophatcat01,Reminder for anyone who wants to come there’s ...,2026-04-05T19:58:09.369000+00:00,"[-0.02591967023909092, -0.021378491073846817, ..."
1,1488692176413266150,lazyboy395,The only issue is my parents are...well...my p...,2026-04-01T00:11:13.420000+00:00,"[-0.041255608201026917, 0.014870340004563332, ..."
2,1488282746475515954,craizo,I usually would but I can't guarantee i'll hav...,2026-03-30T21:04:17.716000+00:00,"[0.013090678490698338, 0.00802773330360651, -0..."
3,1488276656673132654,lazyboy395,I would be down. I live in augusta though. \n\...,2026-03-30T20:40:05.794000+00:00,"[0.009586445055902004, -0.05018388107419014, 0..."
4,1488276435196969192,speedy4134,Is anyone planning on hosting a Fourth of July...,2026-03-30T20:39:12.990000+00:00,"[0.0018161576008424163, -0.014378989115357399,..."


In [4]:
import weaviate.classes as wvc

# Pre-requisite: start Weaviate with Docker before running this cell:
#   docker run -d --rm -p 8081:8080 -p 50052:50051 cr.weaviate.io/semitechnologies/weaviate:latest

client = weaviate.connect_to_local(port=8081, grpc_port=50052)

# --- Create collection ---
# We supply our own vectors, so vectorizer is set to none.
COLLECTION_NAME = "DiscordMessage"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

messages = client.collections.create(
    name=COLLECTION_NAME,
    vectorizer_config=wvc.config.Configure.Vectorizer.none(),
    properties=[
        wvc.config.Property(name="message_id", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="author",     data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="content",    data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="created_at", data_type=wvc.config.DataType.TEXT),
    ],
)

# --- Ingest using pre-computed embeddings from the CSV ---
objects = [
    wvc.data.DataObject(
        properties={
            "message_id": str(row["id"]),
            "author":     str(row["user"]),
            "content":    str(row["content"]),
            "created_at": str(row["timestamp"]),
        },
        vector=row["embedding"],
    )
    for _, row in df.iterrows()
]

result = messages.data.insert_many(objects)
print(f"Inserted {len(objects)} messages. Errors: {len(result.errors)}")


d:\Users\Daniel Hamill\Documents\Projects\discord-llm-agent\history-bot\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Inserted 8228 messages. Errors: 0


In [6]:
query = "I would like to host a new years party."
query_vector = model.encode(query).tolist()

# Hybrid search combines BM25 keyword matching with vector similarity.
results = messages.query.hybrid(
    query=query,
    vector=query_vector,
    limit=5,
    return_metadata=wvc.query.MetadataQuery(score=True),
)

print(f"Query: {query!r}\n")
for i, obj in enumerate(results.objects, 1):
    p = obj.properties
    print(f"[{i}] score={obj.metadata.score:.4f}")
    print(f"     {p['created_at']}  {p['author']}: {p['content']}")
    print()


Query: 'I would like to host a new years party.'

[1] score=1.0000
     2025-11-23T00:28:51.852000+00:00  burritosrule77: Btw, unless there are objections, i would like to host New Years party this year

[2] score=0.9847
     2022-12-06T18:39:32.147000+00:00  crimson7270: I’m probably gonna host a new years party

[3] score=0.6764
     2024-11-06T19:46:56.140000+00:00  boostaslim: I should prolly start planning a new year party

[4] score=0.6261
     2024-11-13T00:10:14.873000+00:00  craizo: Also another question, did we wanna have an actual party on new years in addition to praries? I can host if we need one

[5] score=0.5871
     2022-12-14T23:57:19.061000+00:00  boostaslim: Apologies for the @everyone but would y’all want a New Years party at my place again??



In [ ]:
# Always close the embedded client when done to release the port.
client.close()
